# Analyse de survie en pratique

## Quelques donn?es

On r?cup?re les donn?es disponibles sur *open.data.gouv.fr* [Donn?es hospitali?res relatives ? l'?pid?mie de COVID-19](https://www.data.gouv.fr/fr/datasets/donnees-hospitalieres-relatives-a-lepidemie-de-covid-19/). Ces donn?es ne permettent pas de construire la courbe de [Kaplan-Meier](https://fr.wikipedia.org/wiki/Estimateur_de_Kaplan-Meier). On sait combien de personnes rentrent et sortent chaque jour mais on ne sait pas quand une personne qui sort un 1er avril est entr?e.

In [ ]:
import numpy.random as rnd

import pandas

df = pandas.read_csv(
    "https://www.data.gouv.fr/fr/datasets/r/63352e38-d353-4b54-bfd1-f1b3ee1cabd7",
    sep=";",
)
gr = df[["jour", "rad", "dc"]].groupby(["jour"]).sum()
diff = gr.diff().reset_index(drop=False)
diff.head()

In [ ]:
def donnees_artificielles(hosp, mu=14, nu=21):
    dt = pandas.to_datetime(hosp["jour"])
    res = []
    for i in range(hosp.shape[0]):
        date = dt[i].dayofyear
        h1 = hosp.iloc[i, 1]
        h2 = hosp.iloc[i, 2]
        if h1 < 0 or h2 < 0:
            continue
        delay1 = rnd.exponential(mu, int(h1))
        for j in range(delay1.shape[0]):
            res.append([date - int(delay1[j]), date, 1])
        delay2 = rnd.exponential(mu, int(h2))
        for j in range(delay2.shape[0]):
            res.append([date - int(delay2[j]), date, 0])
    return pandas.DataFrame(res, columns=["entree", "sortie", "issue"])


data = donnees_artificielles(diff[1:].reset_index(drop=True)).sort_values("entree")
data.head()

Chaque ligne est une personne, `entree` est le jour d'entr?e ? l'h?pital, `sortie` celui de la sortie, `issue`, 0 pour d?c?s, 1 pour en vie.

In [ ]:
data.describe()

Il y a environ 80% de survie dans ces donn?es.

In [ ]:
import numpy

duree = data.sortie - data.entree
deces = (data.issue == 0).astype(numpy.int32)

In [ ]:
import numpy
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
kmf = KaplanMeierFitter()
kmf.fit(duree, deces)
kmf.plot(ax=ax)
ax.legend();

## R?gression de Cox

On reprend les donn?es artificiellement g?n?r?es et on ajoute une variable identique ? la dur?e plus un bruit mais quasi nul 

In [ ]:
import pandas

data_simple = pandas.DataFrame(
    {
        "duree": duree,
        "deces": deces,
        "X1": duree * 0.57 * deces + numpy.random.randn(duree.shape[0]),
        "X2": duree * (-0.57) * deces + numpy.random.randn(duree.shape[0]),
    }
)
data_simple.head()

In [ ]:
from sklearn.model_selection import train_test_split

data_train, data_test = train_test_split(data_simple, test_size=0.8)

In [ ]:
from lifelines.fitters.coxph_fitter import CoxPHFitter

cox = CoxPHFitter()
cox.fit(
    data_train[["duree", "deces", "X1"]],
    duration_col="duree",
    event_col="deces",
    show_progress=True,
)

In [ ]:
cox.print_summary()

In [ ]:
cox2 = CoxPHFitter()
cox2.fit(
    data_train[["duree", "deces", "X2"]],
    duration_col="duree",
    event_col="deces",
    show_progress=True,
)
cox2.print_summary()

In [ ]:
cox.predict_cumulative_hazard(data_test[:5])

In [ ]:
cox.predict_survival_function(data_test[:5])